In [44]:
import requests
from bs4 import BeautifulSoup

import pandas as pd
from rapidfuzz import fuzz

import asyncio

# uv add playwright
# uv run playwright install
# uv run playwright install-deps
from playwright.async_api import async_playwright 

In [60]:
def scrape_autokinito(base="https://autokinito.com.cy", HEADERS={"User-Agent": "Mozilla/5.0"}):   
    # Set target page
    url = f"{base}/antiprosopies/"

    # Get the page's HTML
    res = requests.get(url, headers=HEADERS)
    soup = BeautifulSoup(res.text, "html.parser")

    return soup


def get_aftokinito_cars(soup, base="https://autokinito.com.cy", HEADERS={"User-Agent": "Mozilla/5.0"}):
    # Car brands and models are located under the "make/" URL path
    make_links = []

    # Find all <a> tags where the href contains the /make/ path
    for a in soup.select("a[href*='/make/']"):
        make_links.append(base + a["href"])

    cars = []

    # Loop through each brand link
    for link in set(make_links):
        r = requests.get(link, headers=HEADERS)
        s = BeautifulSoup(r.text, "html.parser")

        # Extract brand name
        brand = link.rstrip("/").split("/")[-1].upper()

        # Extract models
        for m in s.select("h2, h3"):
            model = m.get_text(strip=True)
            if model:
                cars.append((brand, model))
    
    return cars
    

def validate_aftokinito_cars(cars):
    valid_models = dict()

    for car in cars:
        brand = car[0].lower().strip()
        model = car[1].lower().strip()
        
        if brand not in valid_models:
            valid_models[brand] = []
        
        # In aftokinito's site, each model starts with its brand, but the way brand is stored in model may slightly differ
        model_prefix = model[:len(brand)]
        similarity = fuzz.ratio(brand, model_prefix)
        
        if similarity >= 70:
            valid_models[brand].append(model)

    return valid_models

    # aftokinito = pd.DataFrame(
    #     [(brand, m) for brand, models_list in models.items() for m in models_list],
    #     columns=["Brand", "Model"]
    # )

    # aftokinito["Brand"].str.title()
    # aftokinito["Model"].str.title()
    # aftokinito["Source"] = "autokinito.com.cy"

    # return aftokinito

In [ ]:
async def scrape_bazaraki(pages=1):
    base_url = "https://www.bazaraki.com/car-motorbikes-boats-and-parts/cars-trucks-and-vans/"
    soups = []
    
    # Blocked by Cloudflare: https://scrapeops.io/web-scraping-playbook/how-to-bypass-cloudflare/ 
    # Bypass using playwright
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,  # important for Cloudflare
            args=["--disable-blink-features=AutomationControlled"]
        )

        context = await browser.new_context(
            user_agent=(
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            viewport={"width": 1280, "height": 900},
            locale="en-US"
        )

        page = await context.new_page()

        for i in range(1, pages + 1):
            url = f"{base_url}?page={i}"

            await page.goto(url, wait_until="domcontentloaded", timeout=60000)

            # wait for page structure 
            await page.wait_for_selector("body")

            await page.wait_for_timeout(3000)  # allow JS rendering

            html = await page.content()
            soups.append(BeautifulSoup(html, "html.parser"))

        await browser.close()

    return soups


def get_bazaraki_cars(soups):
    cars = []
    for soup in soups:

        # Ads in div.advert; brand + model in <a.advert__content-title>
        for advert in soup.select("div.advert"):

            title_tag = advert.select_one("a.advert__content-title")
            if not title_tag:
                continue

            title = title_tag.get_text(strip=True)
            cars.append(title)

    return cars


def validate_bazaraki_cars(cars):
    pass

In [ ]:
# aftokinito_soup = scrape_autokinito()
# aftokinito_cars = get_aftokinito_cars(aftokinito_soup)
# valid_aftokinito_cars = validate_aftokinito_cars(aftokinito_cars)

# bazaraki_soups = await scrape_bazaraki(175)
# bazaraki_cars = get_bazaraki_cars(bazaraki_soups)




# Rapid fuzz to identify existing models

10500